In [1]:
import pandas as pd

data_dir = "../data/raw/nhanes/"

demo = pd.read_sas(
    data_dir + "DEMO_L.xpt",
    format="xport",
    encoding="latin-1"
)

print("Linhas e colunas:", demo.shape)
display(demo.head())

Linhas e colunas: (11933, 27)


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60


In [2]:
print("Participantes:", demo["SEQN"].nunique())
print("SEQN duplicado:", demo["SEQN"].duplicated().sum())
print("Valores ausentes por coluna:")
display(demo.isna().sum().sort_values(ascending=False).head(10))

Participantes: 11933
SEQN duplicado: 0
Valores ausentes por coluna:


RIDAGEMN    11556
RIDEXPRG    10430
DMDYRUSR    10058
DMDHSEDZ     9806
RIDEXAGM     9146
DMDHREDZ     8187
DMDHRMAZ     7913
DMDHRGND     7818
DMDHRAGZ     7809
DMDMARTZ     4141
dtype: int64

In [3]:
bmx = pd.read_sas(
    data_dir + "BMX_L.xpt",
    format="xport",
    encoding="latin-1"
)

print("Formato de BMX:", bmx.shape)
display(bmx.head())

Formato de BMX: (8860, 22)


,SEQN,BMDSTATS,BMXWT,BMIWT,BMXRECUM,BMIRECUM,BMXHEAD,BMIHEAD,BMXHT,BMIHT,...,BMXLEG,BMILEG,BMXARML,BMIARML,BMXARMC,BMIARMC,BMXWAIST,BMIWAIST,BMXHIP,BMIHIP
0,130378.0,1.0,86.9,NaN,NaN,NaN,NaN,NaN,179.5,NaN,...,42.8,NaN,42.0,NaN,35.7,NaN,98.3,NaN,102.9,NaN
1,130379.0,1.0,101.8,NaN,NaN,NaN,NaN,NaN,174.2,NaN,...,38.5,NaN,38.7,NaN,33.7,NaN,114.7,NaN,112.4,NaN
2,130380.0,1.0,69.4,NaN,NaN,NaN,NaN,NaN,152.9,NaN,...,38.5,NaN,35.5,NaN,36.3,NaN,93.5,NaN,98.0,NaN
3,130381.0,1.0,34.3,NaN,NaN,NaN,NaN,NaN,120.1,NaN,...,NaN,NaN,25.4,NaN,23.4,NaN,70.4,NaN,NaN,NaN
4,130382.0,3.0,13.6,NaN,NaN,1.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,NaN


In [4]:
print("Participantes em BMX:", bmx["SEQN"].nunique())
print("SEQN duplicado:", bmx["SEQN"].duplicated().sum())

Participantes em BMX: 8860
SEQN duplicado: 0


In [5]:
base_demo_bmx = demo.merge(
    bmx,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Formato de DEMO:", demo.shape)
print("Formato de BMX:", bmx.shape)
print("Formato após relacionamento:", base_demo_bmx.shape)
print("SEQN duplicado:", base_demo_bmx["SEQN"].duplicated().sum())

Formato de DEMO: (11933, 27)
Formato de BMX: (8860, 22)
Formato após relacionamento: (11933, 48)
SEQN duplicado: 0


In [6]:
colunas_bmx = [
    "BMXWT",
    "BMXHT",
    "BMXBMI",
    "BMXWAIST"
]

display(
    base_demo_bmx[colunas_bmx]
    .isna()
    .sum()
    .to_frame("quantidade_nulos")
)

display(
    (
        base_demo_bmx[colunas_bmx]
        .isna()
        .mean()
        .mul(100)
        .round(2)
        .to_frame("percentual_nulos")
    )
)

has_bmx = base_demo_bmx["BMXBMI"].notna()

print("Com IMC:", has_bmx.sum())
print("Sem IMC:", has_bmx.sum() == False)

print("Sem IMC:", (~has_bmx).sum())

,quantidade_nulos
BMXWT,3179
BMXHT,3434
BMXBMI,3462
BMXWAIST,3743


,percentual_nulos
BMXWT,26.64
BMXHT,28.78
BMXBMI,29.01
BMXWAIST,31.37


Com IMC: 8471
Sem IMC: False
Sem IMC: 3462


In [11]:
print("base_demo_bmx exists:", "base_demo_bmx" in globals())
print("RIAGENDR exists:", "RIAGENDR" in base_demo_bmx.columns)
print("Rows:", base_demo_bmx.shape[0])

print(
    base_demo_bmx["RIAGENDR"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

base_demo_bmx exists: True
RIAGENDR exists: True
Rows: 11933
RIAGENDR
1.0    5575
2.0    6358


In [12]:
gender_counts = (
    base_demo_bmx["RIAGENDR"]
    .value_counts(dropna=False)
    .sort_index()
)

display(gender_counts)


RIAGENDR
1.0    5575
2.0    6358
Name: count, dtype: int64

In [13]:
female_participants = base_demo_bmx.loc[
    base_demo_bmx["RIAGENDR"] == 2.0
    ].copy()

print("Female participants:", female_participants.shape[0])
print(
    "Female participants with BMI:",
    female_participants["BMXBMI"].notna().sum()
)
print(
    "Female participants without BMI:",
    female_participants["BMXBMI"].isna().sum()
)

Female participants: 6358
Female participants with BMI: 4534
Female participants without BMI: 1824


In [14]:
print(
    female_participants["RIDAGEYR"]
    .describe()
    .to_string()
)

count    6.358000e+03
mean     3.936537e+01
std      2.530851e+01
min      5.397605e-79
25%      1.500000e+01
50%      3.900000e+01
75%      6.300000e+01
max      8.000000e+01


In [17]:
mcq = pd.read_sas(
    data_dir + "MCQ_L.xpt",
    format="xport",
    encoding="latin-1"
)

print("MCQ shape:", mcq.shape)
print("Number of participants:", mcq["SEQN"].nunique())
print("Duplicated SEQN:", mcq["SEQN"].duplicated().sum())

print(mcq.columns)

print(mcq.head())

MCQ shape: (11744, 35)
Number of participants: 11744
Duplicated SEQN: 0
Index(['SEQN', 'MCQ010', 'MCQ035', 'MCQ040', 'MCQ050', 'AGQ030', 'MCQ053',
       'MCQ149', 'MCQ160A', 'MCQ195', 'MCQ160B', 'MCQ160C', 'MCQ160D',
       'MCQ160E', 'MCQ160F', 'MCQ160M', 'MCQ170M', 'MCQ160P', 'MCQ160L',
       'MCQ170L', 'MCQ500', 'MCQ510A', 'MCQ510B', 'MCQ510C', 'MCQ510D',
       'MCQ510E', 'MCQ510F', 'MCQ550', 'MCQ560', 'MCQ220', 'MCQ230A',
       'MCQ230B', 'MCQ230C', 'MCQ230D', 'OSQ230'],
      dtype='str')
       SEQN  MCQ010  MCQ035  MCQ040  MCQ050  AGQ030  MCQ053  MCQ149  MCQ160A  \
0  130378.0     2.0     NaN     NaN     NaN     2.0     2.0     NaN      1.0   
1  130379.0     2.0     NaN     NaN     NaN     2.0     2.0     NaN      2.0   
2  130380.0     2.0     NaN     NaN     NaN     2.0     2.0     NaN      2.0   
3  130381.0     2.0     NaN     NaN     NaN     1.0     2.0     NaN      NaN   
4  130382.0     2.0     NaN     NaN     NaN     2.0     2.0     NaN      NaN   

   MCQ195  ...  

In [18]:
female_medical = female_participants.merge(
    mcq,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Female participants:", female_participants.shape[0])
print("Female medical shape:", female_medical.shape)
print(
    "Duplicated SEQN:",
    female_medical["SEQN"].duplicated().sum()
)

Female participants: 6358
Female medical shape: (6358, 82)
Duplicated SEQN: 0


In [19]:
mcq_columns = [
    column
    for column in mcq.columns
    if column != "SEQN"
]

has_mcq_data = female_medical[mcq_columns].notna().any(axis=1)

print("Female participants with MCQ data:", has_mcq_data.sum())
print("Female participants without MCQ data:", (~has_mcq_data).sum())

Female participants with MCQ data: 6274
Female participants without MCQ data: 84


In [21]:
rhq = pd.read_sas(
    data_dir + "RHQ_L.xpt",
    format="xport",
    encoding="latin-1"
)

print("RHQ shape:", rhq.shape)
print("Number of participants:", rhq["SEQN"].nunique())
print("Duplicated SEQN:", rhq["SEQN"].duplicated().sum())
print("Columns:", rhq.columns.tolist())

RHQ shape: (3917, 13)
Number of participants: 3917
Duplicated SEQN: 0
Columns: ['SEQN', 'RHQ010', 'RHQ031', 'RHD043', 'RHQ060', 'RHQ078', 'RHQ131', 'RHD143', 'RHD167', 'RHQ200', 'RHD280', 'RHQ305', 'RHQ332']


In [22]:
female_health = female_medical.merge(
    rhq,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Previous shape:", female_medical.shape)
print("New shape:", female_health.shape)
print(
    "Duplicated SEQN:",
    female_health["SEQN"].duplicated().sum()
)

Previous shape: (6358, 82)
New shape: (6358, 94)
Duplicated SEQN: 0


In [23]:
rhq_columns = [
    column
    for column in rhq.columns
    if column != "SEQN"
]

has_rhq_data = female_health[rhq_columns].notna().any(axis=1)

print("Female participants with RHQ data:", has_rhq_data.sum())
print("Female participants without RHQ data:", (~has_rhq_data).sum())

Female participants with RHQ data: 3384
Female participants without RHQ data: 2974
